# Weights & Biases Experiments Notebook
---

In [5]:
import wandb
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from models.classification import VGG11Classifier
from models.localization import VGG11Localizer
from models.segmentation import VGG11UNet
from data.pets_dataset import OxfordIIITPetDataset
from losses.iou_loss import IoULoss

from sklearn.metrics import f1_score
import numpy as np


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_ds = OxfordIIITPetDataset('./data', 'trainval', 224)
val_ds   = OxfordIIITPetDataset('./data', 'test', 224)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=64)

[PetDataset] split=trainval | samples=3680
[PetDataset] split=test | samples=3669


## Section 2.1 — BatchNorm effect on activations

In [ ]:
def run_bn_experiment(use_bn=True):
    wandb.init(project='da6401-assignment2', name=f'bn_{use_bn}')

    model = VGG11Classifier(num_classes=37, use_bn=use_bn).to(device)
    model.eval()

    batch = next(iter(train_loader))
    imgs = batch['image'].to(device)

    activations = []

    def hook_fn(module, inp, out):
        activations.append(out.detach().cpu().flatten())

    handle = model.encoder.block2[0][0].register_forward_hook(hook_fn)

    with torch.no_grad():
        model(imgs)

    handle.remove()

    act = torch.cat(activations).numpy()

    wandb.log({
        'activation_hist': wandb.Histogram(act),
        'mean': act.mean(),
        'std': act.std()
    })

    wandb.finish()


# run_bn_experiment(True)
# run_bn_experiment(False)

[PetDataset] split=trainval | samples=3680


activation_mean,▁
activation_std,▁
activation_mean,0.35006
activation_std,0.50309


activation_mean,▁
activation_std,▁
activation_mean,-0.09014
activation_std,2.13566
